In [ ]:
from pathlib import Path
from glob import glob

import numpy as np
from IPython.display import display
import ipywidgets as widgets

from dengine.analysis import find_confusion_matrices

experiments = glob("/path/to/logs" + '/**/config.yaml', recursive=True)
experiments = [Path(x).parent for x in experiments]

rows = []
for exp in experiments:
    cm = find_confusion_matrices(exp)
    rounds, _, _ = np.load(cm['confusion_matrix_tr.npy'][0]).shape
    rows.append((exp.name, rounds))

rows.sort(key=lambda x: x[1], reverse=True)

In [ ]:
max_rounds = max(r for _, r in rows) if rows else 1


def bar(rounds, max_r):
    pct = rounds / max_r * 100
    return f"""
    <div style='display:flex;align-items:center;gap:10px'>
      <div style='flex:1;background:#1a1a2e;border-radius:2px;height:6px;overflow:hidden'>
        <div style='width:{pct:.1f}%;background:linear-gradient(90deg,#6c63ff,#a78bfa);height:100%;border-radius:2px'></div>
      </div>
      <span style='font-variant-numeric:tabular-nums;color:#a78bfa;font-size:13px;min-width:36px;text-align:right'>{rounds}</span>
    </div>"""


row_html = ""
for i, (name, rounds) in enumerate(rows):
    bg = "background:rgba(255,255,255,0.02)" if i % 2 == 0 else ""
    row_html += f"""
    <tr style='{bg}border-bottom:1px solid rgba(255,255,255,0.05)'>
      <td style='padding:11px 20px;font-size:12px;color:#cbd5e1;font-family:monospace;letter-spacing:0.3px'>{name}</td>
      <td style='padding:11px 20px;min-width:200px'>{bar(rounds, max_rounds)}</td>
    </tr>"""


html = f"""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&display=swap');
  .exp-dash * {{ box-sizing:border-box; margin:0; padding:0 }}
</style>
<div class='exp-dash' style='
  background:linear-gradient(135deg,#0f0f1a 0%,#16112b 100%);
  border:1px solid rgba(108,99,255,0.25);
  border-radius:16px;
  overflow:hidden;
  font-family:"Space Mono",monospace;
  max-width:780px;
'>
  <table style='width:100%;border-collapse:collapse'>
    <thead>
      <tr style='border-bottom:1px solid rgba(108,99,255,0.2)'>
        <th style='text-align:left;padding:14px 20px;font-size:10px;color:#6c63ff;letter-spacing:1.5px;font-weight:700'>EXPERIMENT ({len(rows)})</th>
        <th style='text-align:left;padding:14px 20px;font-size:10px;color:#6c63ff;letter-spacing:1.5px;font-weight:700'>ROUNDS</th>
      </tr>
    </thead>
    <tbody>{row_html}</tbody>
  </table>
</div>
"""

display(widgets.HTML(html))